# 02 – Vector Search index
Creates a Vector Search endpoint and a Delta Sync index on the policy chunks table, using `gte-large-en` for embeddings.

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
CATALOG = "workspace"
SCHEMA = "policy_rag"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.policy_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.policy_chunks_index"
ENDPOINT_NAME = "policy_rag_endpoint"

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

existing = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if ENDPOINT_NAME in existing:
    print(f"Endpoint '{ENDPOINT_NAME}' already exists")
else:
    vsc.create_endpoint_and_wait(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    print(f"Endpoint '{ENDPOINT_NAME}' created")

In [0]:
EMBEDDING_ENDPOINT = "databricks-gte-large-en"

existing_indexes = [i["name"] for i in vsc.list_indexes(ENDPOINT_NAME).get("vector_indexes", [])]
if INDEX_NAME in existing_indexes:
    print(f"Index '{INDEX_NAME}' already exists")
else:
    vsc.create_delta_sync_index_and_wait(
        endpoint_name=ENDPOINT_NAME,
        index_name=INDEX_NAME,
        source_table_name=CHUNKS_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
    )
    print(f"Index '{INDEX_NAME}' created")

In [0]:
index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

status = index.describe()["status"]
print("Ready:", status.get("ready"))
print("Indexed rows:", status.get("indexed_row_count"))

In [0]:
test_questions = [
    "How many vacation days do I get?",
    "Can I stay home when my kid is sick?",
    "Can I paste a customer contract into Microsoft 365 Copilot?",
    "Who do I contact if my payslip is wrong?",
    "What is the company's pension plan?",  # not covered by any policy at the moment
]

for question in test_questions:
    results = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "document", "section"],
        num_results=3,
    )
    print(f"\nQ: {question}")
    for chunk_id, document, section, score in results["result"]["data_array"]:
        print(f"   {score:.3f}  {chunk_id}  ({document} / {section})")